In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/My Drive/Colab Notebooks

/content/drive/My Drive/Colab Notebooks


In [ ]:
import os
import time
import cv2
import numpy as np
from model.yolo_model import YOLO

In [ ]:
def process_image(img):
  """
  Resize, reduce and expand image
  Agrument: image: the original image
  return image_org:ndarray(64,64,3), processed image
  """
  image_org = cv2.resize(img, (416,416), interpolation = cv2.INTER_CUBIC)
  image_org = np.array(image_org, dtype='float32')
  image_org /=255
  image_org = np.expand_dims(image_org, axis=0)
  return image_org

In [ ]:
# the class function will help us to grab classes from the classes text file
# The COCO dataset contains almost 80 classes
def get_classes(file):
  """
  get classes name.
  file: classes name for database
  class_names: list, classes name
  """
  with open(file) as f:
    name_of_class= f.readlines()
    name_of_class_names = [c.strip() for c in name_of_class]
    return name_of_class

In [ ]:
def box_draw(image, boxes, scores, classes, all_classes):
    """Draw the boxes on the image.

    # Argument:
        image: original image.
        boxes: ndarray, boxes of objects.
        classes: ndarray, classes of objects.
        scores: ndarray, scores of objects.
        all_classes: all classes name.
    """
    for box, score, cl in zip(boxes, scores, classes):
        x, y, w, h = box

        top = max(0, np.floor(x + 0.5).astype(int))
        left = max(0, np.floor(y + 0.5).astype(int))
        right = min(image.shape[1], np.floor(x + w + 0.5).astype(int))
        bottom = min(image.shape[0], np.floor(y + h + 0.5).astype(int))

        cv2.rectangle(image, (top, left), (right, bottom), (255, 0, 0), 2)
        cv2.putText(image, '{0} {1:.2f}'.format(all_classes[cl], score),
                    (top, left - 6),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0, 0, 255), 1,
                    cv2.LINE_AA)

        print('class: {0}, score: {1:.2f}'.format(all_classes[cl], score))
        print('box coordinate x,y,w,h: {0}'.format(box))

    print()

In [ ]:
def detect_image(image, yolo, all_classes):
    """Use yolo v3 to detect images.

    # Argument:
        image: original image.
        yolo: YOLO, yolo model.
        all_classes: all classes name.

    # Returns:
        image: processed image.
    """
    pimage = process_image(image)

    start = time.time()
    image_boxes, image_classes, image_scores = yolo.predict(pimage, image.shape)
    end = time.time()

    print('time: {0:.2f}s'.format(end - start))

    if image_boxes is not None:
        box_draw(image, image_boxes, image_scores, image_classes, all_classes)

    return image

In [ ]:
from google.colab.patches import cv2_imshow

In [ ]:
yolo = YOLO(0.6,0.5)
file = 'data/coco_classes.txt'
all_classes = get_classes(file)

In [ ]:
f = 'bike2.JPG'
path = 'images3/test/bike2.JPG'
image = cv2.imread(path)


In [ ]:
print(image)

[[[141 100  31]
  [140  99  30]
  [139  98  29]
  ...
  [185 160 120]
  [185 160 120]
  [185 160 120]]

 [[140  99  30]
  [140  99  30]
  [140  99  30]
  ...
  [185 160 120]
  [185 160 120]
  [185 160 120]]

 [[140  99  30]
  [140  99  30]
  [141 100  31]
  ...
  [185 160 120]
  [185 160 120]
  [185 160 120]]

 ...

 [[135 150 153]
  [136 151 154]
  [137 152 155]
  ...
  [122 134 128]
  [119 131 125]
  [114 126 120]]

 [[135 150 152]
  [135 150 152]
  [135 150 152]
  ...
  [122 134 128]
  [119 131 125]
  [114 126 120]]

 [[136 151 153]
  [135 150 152]
  [134 149 151]
  ...
  [122 134 128]
  [120 132 126]
  [114 126 120]]]


In [ ]:
image =detect_image(image, yolo, all_classes)
cv2.imwrite('images3/res/' + f, image)

time: 1.25s
class: person
, score: 0.95
box coordinate x,y,w,h: [315.04174781  40.22873774 158.41731679 194.25206083]
class: motorbike
, score: 1.00
box coordinate x,y,w,h: [ 33.36813498 137.87935218 345.52435446 353.75763434]
class: motorbike
, score: 0.77
box coordinate x,y,w,h: [306.30271339  75.13220148 169.18367553 228.04016832]



True